# Introduction to DataFrames

{doc}`Spark Runtime Architecture <../01_apache_spark_runtime_architecture/02_spark_runtime_architecture>`
showed that the driver turns your code into a query plan and the executors
run it. The **DataFrame** is the API you use to write that code. This page
covers what a DataFrame is, how to build and transform one, and why nothing
actually runs until you ask for a result.

The examples use the course's session, created as in
{ref}`Starting a Session <spark-session-start>`:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

## What a DataFrame Is

A DataFrame is a distributed collection of data organised into **named
columns**
([Apache Spark: Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)),
like a SQL table or a spreadsheet. If you know pandas, the shape will feel
familiar, but there are three important differences:

- **It is distributed.** The rows are split into *partitions* spread across
  the executors. No single machine needs to hold the whole thing.
- **It is immutable.** Methods that transform a DataFrame return a *new*
  DataFrame
  ([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#chaining-calls)).
  Nothing is modified in place, so you must assign the result to a variable.
- **It is lazy.** Spark does not act on transformations until an action is
  called
  ([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#chaining-calls)).
  A DataFrame holds a *plan* for producing data, not the data itself.

Every DataFrame has three parts:

- **Schema**: the column names and their data types. It can be declared
  explicitly, read from a source such as a table, or inferred from data.
- **Rows**: records, represented in Python as `pyspark.sql.Row` objects.
- **Columns**: expressions over the rows, represented as
  `pyspark.sql.Column` objects. A column can be a simple value (`string`,
  `int`, `double`, `timestamp`, ...) or a complex one (`array`, `map`,
  `struct`, or the semi-structured `variant` type added in Spark 4.0
  ([Apache Spark: Spark Release 4.0.0](https://spark.apache.org/releases/spark-release-4-0-0.html))).

## Creating a DataFrame

In practice most DataFrames come from tables or files (see
{doc}`Reading and Writing Data <02_reading_and_writing_data>`). For learning
and testing, it is handy to build a small one from Python data with
`createDataFrame`. It infers the schema from the Python objects by default,
and you can pass a schema to set the types yourself
([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#create-a-dataframe-with-specified-values)):

In [2]:
employees = spark.createDataFrame(
    [
        ("Alice", "Engineering", 85000),
        ("Bob", "Engineering", 72000),
        ("Carla", "Sales", 64000),
        ("Dan", "Sales", 58000),
        ("Eve", "Marketing", 70000),
    ],
    schema="name STRING, department STRING, salary INT",
)

employees.printSchema()

root
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



The `schema` argument is a DDL string, the same column syntax you would use
in `CREATE TABLE`. You can also build schemas from `pyspark.sql.types`
(`StructType`, `StructField`, ...), which is more verbose but easier to
generate programmatically.

To look at the data, use `show()`. In a Databricks notebook you can also call
`display(df)`, which renders a results table that you can turn into
visualizations
([Visualizations in Databricks notebooks](https://docs.databricks.com/aws/en/visualizations/#generate-a-result-set-to-visualize)).

In [3]:
employees.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|Alice|Engineering| 85000|
|  Bob|Engineering| 72000|
|Carla|      Sales| 64000|
|  Dan|      Sales| 58000|
|  Eve|  Marketing| 70000|
+-----+-----------+------+



## Working with Columns

Most DataFrame methods take **column expressions**. There are several
equivalent ways to refer to a column:

```python
employees.select("salary")              # a column name as a string
employees.select(F.col("salary"))       # an explicit Column object
employees.select(employees["salary"])   # bound to this specific DataFrame
employees.select(employees.salary)      # attribute access (breaks on names with spaces)
```

`F.col(...)` is the most common choice in production code. The `col` function
lives in the `pyspark.sql.functions` module
([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#select-columns)),
conventionally imported as `F`, which also holds the built-in functions for
strings, dates, arrays, aggregation and more. Column objects support
arithmetic, comparison and boolean operators:

In [4]:
employees.select(
    F.upper("name").alias("name_upper"),
    (F.col("salary") * 1.1).alias("salary_after_raise"),
).show()

+----------+------------------+
|name_upper|salary_after_raise|
+----------+------------------+
|     ALICE| 93500.00000000001|
|       BOB|           79200.0|
|     CARLA|           70400.0|
|       DAN| 63800.00000000001|
|       EVE|           77000.0|
+----------+------------------+



Multiplying by the Python float `1.1` makes the result a `double`, which is
why some values show binary floating-point noise such as `93500.00000000001`.
Round with `F.round` or cast to a `DECIMAL` when exact output matters.

:::{admonition} Note: Prefer Built-in Functions over Python UDFs
:class: note

Built-in Apache Spark functions are optimised for distributed processing and
perform better at scale. A Python user-defined function (UDF) has to
serialise the data and move it out of the JVM to the Python interpreter
([User-defined functions (UDFs)](https://docs.databricks.com/aws/en/udf/#performance-considerations)),
and Photon doesn't run UDFs
([What is Photon?](https://docs.databricks.com/aws/en/compute/photon)). Use a
UDF only for logic that is difficult to express with built-in functions
([User-defined functions (UDFs)](https://docs.databricks.com/aws/en/udf/#when-to-use-a-udf-vs-apache-spark-function)).
:::

## Transformations

A **transformation** takes a DataFrame and returns a new one. These are the
ones you will use constantly:

| Method | What it does | SQL equivalent |
| --- | --- | --- |
| `select(...)` | Choose or compute columns | `SELECT` |
| `withColumn(name, expr)` | Add or replace one column | `SELECT *, expr AS name` |
| `withColumnRenamed(old, new)` | Rename a column | `SELECT old AS new` |
| `drop(...)` | Remove columns | `SELECT` without them |
| `filter(cond)` / `where(cond)` | Keep matching rows | `WHERE` |
| `groupBy(...).agg(...)` | Aggregate per group | `GROUP BY` |
| `join(other, on, how)` | Combine two DataFrames | `JOIN` |
| `orderBy(...)` / `sort(...)` | Sort rows | `ORDER BY` |
| `distinct()` / `dropDuplicates(...)` | Remove duplicate rows | `SELECT DISTINCT` |
| `limit(n)` | Keep the first *n* rows | `LIMIT` |
| `union(other)` / `unionByName(other)` | Stack rows, keeping duplicates | `UNION ALL` |

Because each transformation returns a DataFrame, they chain naturally.
Wrapping the chain in parentheses lets you put one step per line:

In [5]:
engineering = (
    employees
    .filter(F.col("department") == "Engineering")
    .withColumn("monthly_salary", F.round(F.col("salary") / 12, 2))
    .select("name", "monthly_salary")
)
engineering.show()

+-----+--------------+
| name|monthly_salary|
+-----+--------------+
|Alice|       7083.33|
|  Bob|        6000.0|
+-----+--------------+



:::::{dropdown} Example: Average Salary per Department
:open:
:color: primary
:icon: pencil
:class-body: example-body

Using the `employees` DataFrame from above, which department pays the most
on average?

**Group, aggregate and sort.**

```python
avg_salary = (
    employees
    .groupBy("department")
    .agg(F.avg("salary").alias("avg_salary"))
)
avg_salary.orderBy(F.desc("avg_salary")).show()
```

The cell below runs this code. Engineering averages
$(85000 + 72000)/2 = 78500$, Sales $(64000 + 58000)/2 = 61000$, and
Marketing's single employee gives $70000$. Note that `avg` returns a `double`
even though `salary` is an `INT`.
:::::

In [6]:
avg_salary = (
    employees
    .groupBy("department")
    .agg(F.avg("salary").alias("avg_salary"))
)
avg_salary.orderBy(F.desc("avg_salary")).show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|   78500.0|
|  Marketing|   70000.0|
|      Sales|   61000.0|
+-----------+----------+



## Actions

An **action** asks Spark for a result, either returning it to your Python
program or writing it somewhere. Only actions make Spark do any work
([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#chaining-calls)).

| Action | Returns |
| --- | --- |
| `show(n)` / `display(df)` | Prints or renders the first rows |
| `count()` | Number of rows, as a Python `int` |
| `collect()` | *All* rows as a list of `Row` objects on the driver |
| `take(n)` / `head(n)` / `first()` | The first rows as `Row` objects |
| `toPandas()` | All rows as a pandas DataFrame on the driver |
| `df.write...save(...)` / `saveAsTable(...)` | Writes the result to storage |

In [7]:
print(employees.count())
print(employees.take(2))

5


[Row(name='Alice', department='Engineering', salary=85000), Row(name='Bob', department='Engineering', salary=72000)]


`toPandas()` collects every record of the DataFrame to the driver program,
so use it only on a small subset of the data
([Convert between PySpark and pandas DataFrames](https://docs.databricks.com/aws/en/pandas/pyspark-pandas-conversion#example)),
such as an aggregate; the same goes for `collect()`. Each action also runs the
plan again: by default Spark recomputes a transformed dataset every time an
action needs it
([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)),
and serverless compute doesn't support the cache APIs that would keep it
around
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#caching-limitations)).
In production pipelines the write is usually the only action.

## Lazy Evaluation

When you call a transformation, Spark only records it. The work happens when
an action runs, and at that point Spark can optimise the *whole* chain
together. Even reading a table is a transformation: Spark does not load data
until an action runs
([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#create-a-dataframe-from-an-uploaded-file)).

In [8]:
trips = spark.read.table("samples.nyctaxi.trips")      # nothing read yet

long_trips = (
    trips
    .filter(F.col("trip_distance") > 10)               # nothing filtered yet
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount")
)

long_trips.count()                                     # now Spark runs a job

1123

Because the optimiser sees the whole query, it reads only the three columns
that are used and pushes the `trip_distance > 10` filter into the table scan.
Delta tables keep per-file statistics, so the scan can skip files that cannot
contain matching rows
([Data skipping](https://docs.databricks.com/aws/en/delta/data-skipping)).
You can inspect the plan Spark chose with `explain()`, which prints only the
physical plan by default; `mode="formatted"` splits it into an outline and
node details
([Apache Spark API reference: DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)).
In the output below, look for `ReadSchema` with the three columns and the
filter on `trip_distance`:

In [9]:
long_trips.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet samples.nyctaxi.trips (1)


(1) PhotonScan parquet samples.nyctaxi.trips
Output [3]: [tpep_pickup_datetime#10564, trip_distance#10566, fare_amount#10567]
DictionaryFilters: [(trip_distance#10566 > 10.0)]
Location: PreparedDeltaFileIndex [s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b622f329-0f39-444f-9dbb-5733ba3e2021/tables/78b679bd-5656-4852-b0d3-1b93d1ee9e19]
ReadSchema: struct<tpep_pickup_datetime:timestamp,trip_distance:double,fare_amount:double>
RequiredDataFilters: [isnotnull(trip_distance#10566), (trip_distance#10566 > 10.0)]

(2) PhotonColumnarToRow
Input [3]: [tpep_pickup_datetime#10564, trip_distance#10566, fare_amount#10567]

(3) PhotonResultStage
Input [3]: [tpep_pickup_datetime#10564, trip_distance#10566, fare_amount#10567]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing 

Learning to read these plans pays off when diagnosing performance problems.
{doc}`Distributed Systems Programming Fundamentals <../03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`
explains what the operators in them mean.

:::{admonition} Note: Lazy Analysis under Spark Connect
:class: note

On serverless compute and Databricks Connect, Spark defers *checking* your
plan as well as executing it. A misspelt column name is reported when you
first access `df.columns` or `df.schema`, or run an action, not on the line
that contains the typo
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#trigger-eager-analysis-for-error-detection)).
See
{ref}`the Spark Connect section <spark-connect-practical-consequences>` of the
architecture page.
:::

## DataFrames and SQL Are the Same Engine

Anything you can write with the DataFrame API you can also write in SQL, and
the same execution engine computes the result either way
([Apache Spark: Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)).
You can switch between them freely. Register a DataFrame as a temporary view
to query it with SQL; the result of `spark.sql` is an ordinary DataFrame:

In [10]:
employees.createOrReplaceTempView("employees")

top_departments = spark.sql("""
    SELECT department, avg(salary) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

top_departments.filter(F.col("avg_salary") > 65000).show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|   78500.0|
|  Marketing|   70000.0|
+-----------+----------+



Unity Catalog tables can be queried directly by their three-level name
without registering anything:

In [11]:
spark.sql("SELECT count(*) AS n FROM samples.nyctaxi.trips").show()

+-----+
|    n|
+-----+
|21932|
+-----+



`spark.sql` also accepts **parameter markers**. A named marker such as
`:min_distance` gets its value from the `args` dictionary, which keeps values
separate from the structure of the statement and protects your code from SQL
injection
([Parameter markers](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-parameter-marker#named-parameter-markers)):

In [12]:
spark.sql(
    "SELECT count(*) AS n FROM samples.nyctaxi.trips WHERE trip_distance > :min_distance",
    args={"min_distance": 10},
).show()

+----+
|   n|
+----+
|1123|
+----+



The count matches `long_trips.count()` above, because it is the same query
written in SQL.

Inside a DataFrame chain, `F.expr("...")` and `selectExpr("...")` let you use
SQL expressions where that is shorter
([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#select-columns)):

In [13]:
employees.selectExpr("name", "salary * 12 AS yearly_cost").show()

+-----+-----------+
| name|yearly_cost|
+-----+-----------+
|Alice|    1020000|
|  Bob|     864000|
|Carla|     768000|
|  Dan|     696000|
|  Eve|     840000|
+-----+-----------+



## When to Use the DataFrame API and When to Use SQL

Because both compile to the same engine, the choice is about the code, not
about speed:

| Use | When |
| --- | --- |
| SQL (`spark.sql`, SQL cells) | Ad hoc analysis, queries shared with SQL users, logic that reads naturally as one statement |
| DataFrame API | Logic built up step by step in Python, reused in functions, or unit tested |
| `selectExpr` / `F.expr` | A single expression that is shorter in SQL inside a DataFrame chain |

## Common Mistakes

- **Forgetting that DataFrames are immutable.** Calling a transformation
  without assigning the result does nothing to the original DataFrame,
  because transformations return a new DataFrame
  ([PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics#chaining-calls)).
  The cell below adds a column but throws the result away:

In [14]:
employees.withColumn("bonus", F.col("salary") * 0.1)  # result discarded
employees.columns

['name', 'department', 'salary']

  The fix is to assign it: `employees = employees.withColumn(...)`, or chain
  the next step onto it.
- **Collecting a large DataFrame to the driver.** `collect()` and
  `toPandas()` bring every row to the single driver. Filter and aggregate in
  Spark first
  ([Convert between PySpark and pandas DataFrames](https://docs.databricks.com/aws/en/pandas/pyspark-pandas-conversion#example)).
- **Writing a Python UDF for something a built-in function already does.**
  Python UDFs move data out of the JVM to Python and back, while built-in
  functions are optimised for distributed processing
  ([User-defined functions (UDFs)](https://docs.databricks.com/aws/en/udf/#performance-considerations)).
  Search `pyspark.sql.functions` first.
- **Formatting values into SQL strings.** Building a query with an f-string
  such as `f"... WHERE name = '{user_input}'"` opens the door to SQL injection.
  Use parameter markers instead
  ([Parameter markers](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-parameter-marker)).
- **Using `union` on DataFrames whose columns are in a different order.**
  `union` resolves columns by position, not by name, and keeps duplicates
  ([Apache Spark API reference: DataFrame.union](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.union.html)).
  Use `unionByName` when the column order may differ.

Next, {doc}`Reading and Writing Data <02_reading_and_writing_data>` replaces
these hand-built DataFrames with real tables and files.

## Resources

- [PySpark DataFrame quickstart (Apache Spark)](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)
- [PySpark basics (Databricks)](https://docs.databricks.com/aws/en/pyspark/basics)
- [Tutorial: Load and transform data using Apache Spark DataFrames (Databricks)](https://docs.databricks.com/aws/en/getting-started/dataframes)